## Parameters

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("model_catalog", "")
dbutils.widgets.text("model_schema", "")
dbutils.widgets.text("model_name", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
table_name = dbutils.widgets.get("table_name").strip()
model_catalog = dbutils.widgets.get("model_catalog").strip()
model_schema = dbutils.widgets.get("model_schema").strip()
model_name = dbutils.widgets.get("model_name").strip()

params = {
    "catalog": catalog,
    "schema": schema,
    "table_name": table_name,
    "model_catalog": model_catalog,
    "model_schema": model_schema,
    "model_name": model_name,
}
if not all(params.values()):
    missing = ", ".join(name for name, value in params.items() if not value)
    raise ValueError(f"All parameters must be non-empty; missing: {missing}")

full_table = f"{catalog}.{schema}.{table_name}"
full_model_name = f"{model_catalog}.{model_schema}.{model_name}"

## Load data

In [ ]:
pdf = spark.table(full_table).toPandas()

y = pdf["label"]
X = pdf[[c for c in pdf.columns if c.startswith("feature_")]]

print(f"Loaded {len(pdf)} rows with {X.shape[1]} features from {full_table}")

## Train & convert to ONNX

In [ ]:
import os
import tempfile

import numpy as np
import onnx
import torch
import torch.nn as nn

HIDDEN = 15
EPOCHS = 10
BATCH_SIZE = 4096
LEARNING_RATE = 0.001
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)


class SimpleMLP(nn.Module):
    """Six Linear layers: five hidden (15 units) + one output (1 unit)."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)


X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

model = SimpleMLP(input_dim=X.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.BCELoss()

model.train()
for _ in range(EPOCHS):
    permutation = torch.randperm(X_tensor.size(0))
    for start in range(0, X_tensor.size(0), BATCH_SIZE):
        idx = permutation[start : start + BATCH_SIZE]
        batch_x = X_tensor[idx]
        batch_y = y_tensor[idx]
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = loss_fn(preds, batch_y)
        loss.backward()
        optimizer.step()

model.eval()
onnx_dir = tempfile.mkdtemp()
onnx_path = os.path.join(onnx_dir, "model.onnx")

dummy_input = torch.randn(1, X.shape[1], dtype=torch.float32)
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["float_input"],
    output_names=["output"],
    dynamic_axes={
        "float_input": {0: "batch_size"},
        "output": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print(
    f"Trained SimpleMLP (6 Linear layers, {HIDDEN} hidden units) "
    f"on {len(X)} rows and converted to ONNX"
)

## Register in UC Model Registry

In [ ]:
import mlflow
import numpy as np
import onnxruntime as ort
import pandas as pd
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

mlflow.set_registry_uri("databricks-uc")


class OnnxSigmoidPyFunc(mlflow.pyfunc.PythonModel):
    def __init__(self, model_name: str):
        self.model_name = model_name

    def load_context(self, context):
        self.session = ort.InferenceSession(
            context.artifacts["onnx_model"],
            providers=["CPUExecutionProvider"],
        )
        self.input_name = self.session.get_inputs()[0].name

    def predict(self, context, model_input):
        if isinstance(model_input, pd.DataFrame):
            input_values = model_input.to_numpy(dtype=np.float32, copy=False)
        else:
            input_values = np.asarray(model_input, dtype=np.float32)

        positive_proba = self.session.run(None, {self.input_name: input_values})[0]
        proba = np.asarray(positive_proba, dtype=np.float64).reshape(-1)
        return pd.DataFrame(
            {
                "target": (proba >= 0.5).astype(np.int64),
                "proba": proba,
                "model_name": self.model_name,
            }
        )


input_schema = Schema([ColSpec("double", name) for name in X.columns])
output_schema = Schema(
    [
        ColSpec("long", "target"),
        ColSpec("double", "proba"),
        ColSpec("string", "model_name"),
    ]
)
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
expected_columns = ["target", "proba", "model_name"]

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=OnnxSigmoidPyFunc(model_name=model_name),
        artifacts={"onnx_model": onnx_path},
        signature=signature,
        input_example=X.head(2),
        pip_requirements=[
            "mlflow",
            "onnxruntime",
            "numpy",
            "pandas",
            "cloudpickle",
        ],
    )
    pyfunc_model = mlflow.pyfunc.load_model(model_info.model_uri)
    sample_prediction = pyfunc_model.predict(X.head(2))
    assert list(sample_prediction.columns) == expected_columns
    assert len(sample_prediction) == 2
    assert sample_prediction["model_name"].tolist() == [model_name, model_name]

    registered = mlflow.register_model(model_info.model_uri, full_model_name)

print(f"Registered {registered.name} version {registered.version}")